# LUAD Growth-Pattern Grading — Training Notebook
**Attention-MIL on frozen UNI features · trains in 5 classes · maps to 3-tier grade at inference**

This notebook trains a small model that reads a lung-adenocarcinoma slide and predicts its **predominant growth pattern** (5 classes), then derives an approximate **3-tier grade** at inference. It is built to run comfortably on an **8 GB GPU** because the heavy vision model (UNI) is *frozen* and used only once, beforehand, to turn each slide into a small bag of feature vectors. The part we train here is tiny.

**Before running this notebook you must already have:**
1. The DHMC slides (`.tif`) and their label file.
2. Per-slide feature files (`.h5`) produced by TRIDENT + UNI (the one-time step below).


## The architecture, in plain terms

**The problem's shape — Multiple Instance Learning (MIL).** Each slide is cut into ~2,000 tiles, but we only have *one* label for the whole slide. We don't know which tiles actually show the predominant pattern. So we have a *bag* of tiles and a single bag label — that's MIL.

**Why not just average the tiles?** Most tiles are fat, blood, whitespace, or normal tissue. Averaging drowns the few diagnostic tiles. Taking the single most-activated tile (max) is the opposite problem — too brittle.

**Attention pooling = the fix.** The model learns a *weight* for every tile ("how grade-relevant is this?"), then takes a **weighted average**. Junk tiles get ~0 weight; diagnostic tiles dominate. Those weights also double as a **heatmap** showing where the model looked.

**Gated attention (the version in this notebook).** The tile scorer has two branches:
- a `tanh` branch — *what is in this tile?*
- a `sigmoid` **gate** — *should I let this tile speak?*

Multiplying them lets the network suppress a tile even if it looks interesting. It beats plain attention on pathology with almost no extra parameters.

**Why frozen UNI features.** UNI is a vision model already trained on 100,000 slides; it turns each 256×256 tile into a 1024-number vector that captures its morphology. With only 143 slides we cannot train a vision model without memorising — so UNI does the *seeing*, and our tiny attention head only learns to *focus and vote*. Few trainable parameters = little to overfit.

**5 → 3 at inference.** We train/evaluate on the 5 patterns. To report a 3-tier grade we **sum the probabilities** of grouped classes (lepidic→G1; acinar+papillary→G2; micropapillary+solid→G3). Summing beliefs is more principled than relabelling the single top guess.

**Memory note (why 8 GB is plenty *here*).** Training never touches the giant images — only the small feature bags. One bag (≈2,000 × 1024 floats ≈ 8 MB) sits on the GPU at a time; the model is a few hundred-K parameters. The 8 GB constraint really only mattered during the one-time UNI feature extraction, which TRIDENT handles in batches.


## Step 0 — One-time feature extraction (run in a terminal, not here)

Install TRIDENT and request UNI access on Hugging Face first (see the build playbook). Then turn every slide into an `.h5` bag of UNI features:

```bash
python run_batch_of_slides.py \
    --task all \
    --wsi_dir ./dhmc_slides \
    --job_dir ./dhmc_out \
    --patch_encoder uni_v1 \
    --mag 20 --patch_size 256 --overlap 0
```

This writes one `.h5` per slide (each holding `features` of shape `[N_tiles, 1024]` and usually `coords`). Point `FEATURE_DIR` below at that output folder. ~143 slides take a couple of hours on a 4060 and a few hundred MB of disk.


In [ ]:
# If anything is missing, install it (safe to re-run)
# !pip install torch h5py pandas numpy scikit-learn matplotlib --quiet

import os, glob, h5py, numpy as np, pandas as pd, torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (cohen_kappa_score, f1_score,
                             balanced_accuracy_score, confusion_matrix, accuracy_score)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)
print("Torch:", torch.__version__, "| Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## Step 1 — Configuration

Edit the three paths/column names to match your machine and label file.

In [ ]:
# ---- EDIT THESE ----
FEATURE_DIR = "dhmc_out/features_uni_v1"     # folder of per-slide .h5 files from TRIDENT
LABEL_CSV   = "data/MetaData_Release_1.0.csv"  # the DHMC metadata file (match your exact filename)
SLIDE_COL   = "File Name"                    # confirmed: contains e.g. 'DHMC_0001.tif'
PATTERN_COL = "Class"                        # confirmed: contains 'acinar','solid','lepidic',...
# --------------------

# NOTE on this dataset's class balance (143 slides):
#   acinar 59 | solid 51 | lepidic 19 | micropapillary 9 | papillary 5
# The two rare classes (papillary=5, micropapillary=9) are too sparse for reliable
# 5-class learning -> treat 5-class per-class numbers for them as noisy/diagnostic.
# The mapped 3-tier grade is far more balanced (G1=19, G2=64, G3=60) and is the
# trustworthy headline metric.

# The 5 growth patterns, ordered low -> high aggressiveness (order matters for weighted kappa)
PATTERNS = ["lepidic", "acinar", "papillary", "micropapillary", "solid"]
PAT2ID   = {p: i for i, p in enumerate(PATTERNS)}

# 5-class id -> 3-tier grade id  (0=G1 low, 1=G2 intermediate, 2=G3 high)
ID5_TO_GRADE = {0: 0, 1: 1, 2: 1, 3: 2, 4: 2}   # lepidic->G1 ; acinar,papillary->G2 ; micropap,solid->G3
GRADE_NAMES  = ["G1 (low)", "G2 (intermediate)", "G3 (high)"]

MAX_PATCHES  = 4000   # cap tiles per slide DURING TRAINING (speed + mild regularisation). Eval uses all tiles.
N_FOLDS      = 5
EPOCHS       = 50
LR           = 1e-4
WEIGHT_DECAY = 1e-4
ACCUM        = 8      # gradient-accumulation steps (simulates a bigger batch with batch_size=1)

## Step 2 — Load labels and link each slide to its feature file

We keep only slides that (a) have a valid pattern and (b) actually have an `.h5` feature file on disk. Then we build both the 5-class target (`y5`) and the mapped 3-tier target (`y3`).

**This dataset's reality (confirmed):** the 143 slides are imbalanced — acinar 59, solid 51, lepidic 19, micropapillary 9, papillary 5. The two rare patterns (papillary, micropapillary) are too sparse to learn reliably as standalone 5-class categories, so treat their 5-class numbers as noisy diagnostics. After mapping to the 3-tier grade they balance out nicely (G1=19, G2=64, G3=60) — **the 3-tier grade is your trustworthy headline result.**

In [ ]:
meta = pd.read_csv(LABEL_CSV)

meta["pattern"] = meta[PATTERN_COL].astype(str).str.strip().str.lower()
meta = meta[meta["pattern"].isin(PATTERNS)].copy()
meta["y5"] = meta["pattern"].map(PAT2ID)
meta["y3"] = meta["y5"].map(ID5_TO_GRADE)

# attach the path to each slide's feature file (match by id prefix)
def find_feat(sid):
    # 'File Name' includes the .tif extension; TRIDENT names features by the stem,
    # e.g. 'DHMC_0001.tif' -> feature file 'DHMC_0001.h5'. Strip the extension before matching.
    stem = os.path.splitext(str(sid))[0]
    hits = glob.glob(os.path.join(FEATURE_DIR, f"{stem}*.h5"))
    return hits[0] if hits else None

meta["feat_path"] = meta[SLIDE_COL].astype(str).map(find_feat)
n_missing = meta["feat_path"].isna().sum()
print(f"Slides with no feature file (dropped): {n_missing}")
meta = meta.dropna(subset=["feat_path"]).reset_index(drop=True)

print("Usable slides:", len(meta))
print("\nClass balance (expect imbalance — handled by class weights):")
print(meta["pattern"].value_counts())

## Step 3 — Data loader that is gentle on the GPU

Design choices that keep memory low:
- We read each `.h5` **once into CPU RAM** and cache it (143 small bags ≈ 1–2 GB RAM total). If your RAM is tight, set `cache=False`.
- We move only **one bag at a time** to the GPU inside the training loop — never the whole dataset.
- During **training** we randomly subsample to `MAX_PATCHES` tiles (saves time, adds a little regularisation). During **evaluation** we use *all* tiles for the most faithful prediction.
- The `.h5` feature key is auto-detected, since different tools name it `features` / `feats`.

In [ ]:
def find_feature_key(h5path):
    # TRIDENT/CLAM usually store features under 'features'. Fall back to the first 2D array.
    with h5py.File(h5path, "r") as f:
        for k in ["features", "feats", "feature"]:
            if k in f: return k
        for k in f.keys():
            if getattr(f[k], "ndim", None) == 2:
                return k
    raise KeyError(f"No 2D feature array found in {h5path}")

class SlideBagDataset:
    def __init__(self, df, max_patches=MAX_PATCHES, cache=True):
        self.df = df.reset_index(drop=True)
        self.max_patches = max_patches
        self.cache = cache
        self._mem = {}

    def __len__(self):
        return len(self.df)

    def _read(self, path):
        key = find_feature_key(path)
        with h5py.File(path, "r") as f:
            X = np.asarray(f[key][:], dtype=np.float32)   # [N_tiles, 1024]
        return torch.from_numpy(X)

    def get_bag(self, i, sample=True):
        # returns (features[N,1024] on CPU, y5, y3, slide_id)
        if self.cache and i in self._mem:
            X = self._mem[i]
        else:
            X = self._read(self.df.iloc[i]["feat_path"])
            if self.cache: self._mem[i] = X
        if sample and X.shape[0] > self.max_patches:        # training-time subsample
            idx = torch.randperm(X.shape[0])[:self.max_patches]
            X = X[idx]
        row = self.df.iloc[i]
        return X, int(row.y5), int(row.y3), row[SLIDE_COL]

ds = SlideBagDataset(meta)
# quick sanity check on one slide
_X, _y5, _y3, _sid = ds.get_bag(0, sample=False)
print("Example bag:", _sid, "| tiles x dim =", tuple(_X.shape), "| y5 =", _y5, "| y3 =", _y3)

## Step 4 — The Gated Attention-MIL model

`forward` takes one bag `[N_tiles, 1024]` and returns:
- `logits` `[1, 5]` — the slide's class scores
- `attn` `[N_tiles]` — how much each tile contributed (for heatmaps)

The whole thing is a few hundred-thousand parameters.

In [ ]:
class GatedABMIL(nn.Module):
    def __init__(self, in_dim=1024, hid=256, att_dim=128, n_classes=5, p_drop=0.3):
        super().__init__()
        # project each tile's 1024-dim UNI feature down to a small hidden size
        self.proj  = nn.Sequential(nn.Linear(in_dim, hid), nn.ReLU(), nn.Dropout(p_drop))
        # gated attention: a 'content' branch (tanh) and a 'gate' branch (sigmoid)
        self.att_V = nn.Linear(hid, att_dim)
        self.att_U = nn.Linear(hid, att_dim)
        self.att_w = nn.Linear(att_dim, 1)
        # classifier on the pooled slide vector
        self.head  = nn.Sequential(nn.Dropout(p_drop), nn.Linear(hid, n_classes))

    def forward(self, bag):                              # bag: [N, in_dim]
        h = self.proj(bag)                               # [N, hid]
        gated = torch.tanh(self.att_V(h)) * torch.sigmoid(self.att_U(h))   # [N, att_dim]
        scores = self.att_w(gated)                       # [N, 1]
        attn = torch.softmax(scores, dim=0)              # [N, 1]  weights sum to 1 across tiles
        z = (attn * h).sum(dim=0, keepdim=True)          # [1, hid]  attention-weighted slide vector
        logits = self.head(z)                            # [1, n_classes]
        return logits, attn.squeeze(1)

# count parameters
_m = GatedABMIL()
print("Trainable parameters:", sum(p.numel() for p in _m.parameters()))

## Step 5 — Loss (class-weighted) and prediction / evaluation helpers

Because patterns are imbalanced, we weight the loss so rare classes (lepidic, micropapillary) are not ignored. Evaluation uses **quadratic-weighted Cohen's kappa** as the headline (the classes are ordered, so confusing lepidic↔solid is penalised more than acinar↔papillary), plus macro-F1, balanced accuracy, and confusion matrices.

In [ ]:
def class_weights(labels, n_classes, device):
    counts = np.bincount(labels, minlength=n_classes).astype(np.float32)
    w = counts.sum() / (n_classes * np.clip(counts, 1, None))   # inverse-frequency
    return torch.tensor(w, dtype=torch.float32, device=device)

@torch.no_grad()
def predict_probs(model, ds, indices, device=DEVICE):
    model.eval()
    P5, Y5, Y3, ids = [], [], [], []
    for i in indices:
        X, y5, y3, sid = ds.get_bag(int(i), sample=False)     # eval = use ALL tiles
        logits, _ = model(X.to(device))
        P5.append(torch.softmax(logits, dim=1).cpu().numpy()[0])
        Y5.append(y5); Y3.append(y3); ids.append(sid)
    return np.array(P5), np.array(Y5), np.array(Y3), ids

def probs5_to_3(P5):
    # collapse 5-class probabilities into 3-tier grade probabilities by SUMMING grouped classes
    P3 = np.zeros((P5.shape[0], 3), dtype=np.float32)
    for c5, g in ID5_TO_GRADE.items():
        P3[:, g] += P5[:, c5]
    return P3

def metrics_from_probs(P, Y, n):
    pred = P.argmax(1)
    return dict(
        kappa   = cohen_kappa_score(Y, pred, weights="quadratic", labels=list(range(n))),
        macroF1 = f1_score(Y, pred, average="macro", labels=list(range(n)), zero_division=0),
        balAcc  = balanced_accuracy_score(Y, pred),
        acc     = accuracy_score(Y, pred),
        cm      = confusion_matrix(Y, pred, labels=list(range(n))),
    )

def eval5(P5, Y5): return metrics_from_probs(P5, Y5, 5)
def eval3(P5, Y3): return metrics_from_probs(probs5_to_3(P5), Y3, 3)

## Step 6 — Train one fold

Batch size is **1 slide** (bags have different tile counts), and we **accumulate gradients** over `ACCUM` slides before each optimiser step to mimic a larger, more stable batch. We keep the model from the epoch with the best validation kappa.

In [ ]:
def train_fold(ds, train_idx, val_idx, epochs=EPOCHS, lr=LR, wd=WEIGHT_DECAY,
               accum=ACCUM, device=DEVICE, verbose=False):
    model = GatedABMIL(n_classes=5).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights(ds.df.iloc[train_idx]["y5"].values, 5, device))

    best_kappa, best_state = -1.0, None
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        for step, i in enumerate(np.random.permutation(train_idx)):
            X, y5, _, _ = ds.get_bag(int(i), sample=True)        # train = subsample tiles
            logits, _ = model(X.to(device))
            loss = loss_fn(logits, torch.tensor([y5], device=device)) / accum
            loss.backward()
            if (step + 1) % accum == 0:
                opt.step(); opt.zero_grad()
        opt.step(); opt.zero_grad()                              # flush leftover grads

        P5, Y5, _, _ = predict_probs(model, ds, val_idx, device)
        k = cohen_kappa_score(Y5, P5.argmax(1), weights="quadratic", labels=list(range(5)))
        if k > best_kappa:
            best_kappa = k
            best_state = {kk: v.detach().cpu().clone() for kk, v in model.state_dict().items()}
        if verbose: print(f"  epoch {ep:02d}  val kappa {k:.3f}")
    model.load_state_dict(best_state)
    return model, best_kappa

## Step 7 — 5-fold cross-validation

With only ~143 slides, a single train/test split is too noisy. We run 5 folds, collect **out-of-fold (OOF)** predictions for every slide, and report mean ± std across folds.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_idx = np.arange(len(ds))
y5_all  = ds.df["y5"].values

fold5, fold3 = [], []
oof_P5 = np.zeros((len(ds), 5)); oof_Y5 = np.zeros(len(ds), int); oof_Y3 = np.zeros(len(ds), int)
models = []

for fold, (tr, va) in enumerate(skf.split(all_idx, y5_all)):
    model, vk = train_fold(ds, tr, va)
    models.append(model)
    P5, Y5, Y3, _ = predict_probs(model, ds, va)
    m5, m3 = eval5(P5, Y5), eval3(P5, Y3)
    fold5.append(m5); fold3.append(m3)
    oof_P5[va], oof_Y5[va], oof_Y3[va] = P5, Y5, Y3
    print(f"[fold {fold}] 5-class: kappa={m5['kappa']:.3f} macroF1={m5['macroF1']:.3f} balAcc={m5['balAcc']:.3f}"
          f"  |  3-tier: kappa={m3['kappa']:.3f} acc={m3['acc']:.3f}")

## Step 8 — Results summary (5-class and mapped 3-tier)

Compare the 5-class kappa to the published baseline: the original DHMC model reached kappa ≈ **0.525**, and the pathologists agreed with each other at kappa ≈ **0.485**. Matching or beating ~0.485 is the goal.

In [ ]:
def summarize(folds, name):
    print(f"--- {name} (mean ± std over {N_FOLDS} folds) ---")
    for key in ["kappa", "macroF1", "balAcc", "acc"]:
        v = np.array([f[key] for f in folds])
        print(f"  {key:8s}: {v.mean():.3f} ± {v.std():.3f}")

summarize(fold5, "5-class predominant pattern")
summarize(fold3, "3-tier grade (mapped)")

print("\n5-class out-of-fold confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(confusion_matrix(oof_Y5, oof_P5.argmax(1), labels=range(5)),
                   index=PATTERNS, columns=PATTERNS))

print("\n3-tier out-of-fold confusion matrix:")
print(pd.DataFrame(confusion_matrix(oof_Y3, probs5_to_3(oof_P5).argmax(1), labels=[0,1,2]),
                   index=GRADE_NAMES, columns=GRADE_NAMES))

## Step 9 — The "refer to human" flag (coverage vs accuracy)

A useful tool knows when it's unsure. We rank slides by the model's confidence (top softmax probability) and ask: *if the model only handled the most-confident X% and sent the rest to a pathologist, how accurate would it be on the ones it kept?* Accuracy should climb as coverage drops — that trade-off is the clinical value.

In [ ]:
def coverage_accuracy(P, Y):
    conf = P.max(1); pred = P.argmax(1)
    order = np.argsort(-conf)                      # most confident first
    rows = []
    for frac in np.linspace(0.1, 1.0, 10):
        k = max(1, int(len(Y) * frac))
        idx = order[:k]
        rows.append((round(frac, 2), round((pred[idx] == Y[idx]).mean(), 3)))
    return pd.DataFrame(rows, columns=["coverage", "accuracy_on_kept_slides"])

print("5-class coverage / accuracy trade-off (OOF):")
print(coverage_accuracy(oof_P5, oof_Y5).to_string(index=False))

## Step 10 (optional) — Attention heatmap

If your `.h5` files include tile coordinates (`coords`, which TRIDENT saves), we can paint each tile by its attention weight to show *where* the model looked. Pass any slide's feature path and a trained model.

In [ ]:
import matplotlib.pyplot as plt

@torch.no_grad()
def attention_heatmap(model, h5path, device=DEVICE):
    key = find_feature_key(h5path)
    with h5py.File(h5path, "r") as f:
        X = torch.from_numpy(np.asarray(f[key][:], np.float32))
        coords = np.asarray(f["coords"][:]) if "coords" in f else None
    model.eval()
    _, attn = model(X.to(device))
    a = attn.cpu().numpy()
    if coords is None:
        print("This .h5 has no 'coords' dataset; cannot plot spatially.")
        return
    plt.figure(figsize=(6, 6))
    plt.scatter(coords[:, 0], -coords[:, 1], c=a, s=6, cmap="inferno")
    plt.colorbar(label="attention weight"); plt.axis("equal")
    plt.title("Where the model looked"); plt.tight_layout(); plt.show()

# Example (uncomment):
# attention_heatmap(models[0], meta.iloc[0]["feat_path"])

## Step 11 — Predict a single new slide (inference)

End-to-end inference for one slide: load its features, get the 5-class probabilities, the predicted pattern, and the mapped 3-tier grade.

In [ ]:
@torch.no_grad()
def predict_slide(model, h5path, device=DEVICE):
    key = find_feature_key(h5path)
    with h5py.File(h5path, "r") as f:
        X = torch.from_numpy(np.asarray(f[key][:], np.float32)).to(device)
    model.eval()
    logits, _ = model(X)
    p5 = torch.softmax(logits, dim=1).cpu().numpy()[0]
    p3 = probs5_to_3(p5[None, :])[0]
    return {
        "pattern":      PATTERNS[int(p5.argmax())],
        "pattern_probs": {PATTERNS[i]: round(float(p5[i]), 3) for i in range(5)},
        "grade":        GRADE_NAMES[int(p3.argmax())],
        "grade_probs":  {GRADE_NAMES[i]: round(float(p3[i]), 3) for i in range(3)},
        "confidence":   round(float(p5.max()), 3),
    }

# Example (uncomment):
# import json; print(json.dumps(predict_slide(models[0], meta.iloc[0]["feat_path"]), indent=2))

## Notes, gotchas, and how to make it stronger

- **If kappa is near 0 / model predicts one class:** class imbalance not handled — confirm `class_weights` is applied, and consider training the 3-tier head directly (more balanced).
- **If you run out of RAM caching bags:** set `SlideBagDataset(meta, cache=False)`.
- **If a slide has very few tiles (<100):** it is probably low-tissue; consider dropping it.
- **Save a model:** `torch.save(models[0].state_dict(), "abmil_fold0.pt")`.
- **Stronger validation:** keep a 20% hold-out untouched until the end; or compare UNI vs CONCH vs ResNet50 features (showing UNI > ResNet50 is itself a clean result).
- **Honesty in any writeup:** this predicts the *predominant pattern*; the true IASLC grade needs pattern *proportions*, which the public DHMC labels do not include, so the 3-tier output is an approximation.
